## Desafio 2 — Consumo de Dados (SWAPI) | Globo (Engenharia de Dados)

Este notebook implementa um pipeline simples de **extração → transformação → carga (ETL)** a partir da API pública do Star Wars (SWAPI), armazenando os dados em **SQLite** e gerando **insights via SQL**.

**Stack utilizada**
- Python (requests, pandas)
- SQLite (armazenamento)
- SQL (análises e insights)
- Notebook (Jupyter/Colab)

### 1. Bibliotecas, variáveis e configurações

Nesta seção são importadas as bibliotecas necessárias e definidos os endpoints que serão consumidos.

In [83]:
# Bibliotecas
import sqlite3
import time
from pathlib import Path

import pandas as pd
import requests

In [84]:
# Variáveis
url = 'https://swapi.dev/api'
endpoint = ['films', 'people', 'planets', 'species', 'starships', 'vehicles']

### 2. Funções auxiliares (extração com paginação e extração de ID)

Extração Paginada (`fetch_all`): Esta função percorre todos os links next de um endpoint até consolidar todos os registros em uma única lista, evitando perda de dados.

Tratamento de IDs (`extract_id`): Esta função é responsável por extrair somente o identificador numérico final das URLs para garantir o relacionamento entre os registros.

In [ ]:
def fetch_all(base_url: str, endpoint: str, sleep_s: float = 0.1) -> list[dict]:
    """
    Consome dados da API de forma paginada.
    
    Args:
        base_url: URL base da API (ex: https://swapi.dev/api).
        endpoint: O recurso específico a ser consumido (ex: 'people', 'starships').
        sleep_s: Tempo de espera entre requisições para evitar rate limit.
        
    Returns:
        Uma lista contendo todos os registros (dicts) encontrados no endpoint.
    """
    # Monta a URL inicial combinando a base e o endpoint
    next_url = f"{base_url}/{endpoint}/"
    out = []

    # Itera enquanto houver uma URL de próxima página retornada pela API
    while next_url:
        # Realiza a requisição GET com um limite de tempo (timeout) para evitar travamentos
        r = requests.get(next_url, timeout=30)
        
        # Garante que a função pare caso ocorra um erro de conexão ou permissão (4xx ou 5xx)
        r.raise_for_status()
        
        data = r.json()
        
        # Adiciona os resultados da página atual à lista
        out.extend(data["results"])
        
        # Atualiza a URL para a próxima página ou None, caso chegue ao fim
        next_url = data["next"]
        
        # Pausa controlada para evitar sobrecarga da API
        time.sleep(sleep_s)

    return out

# Função auxiliar para extrair o ID da URL
def extract_id(url_string):
    """
    Extrai o identificador numérico final de uma URL da SWAPI.
    
    Transforma strings no formato 'https://swapi.dev/api/people/1/' em inteiros (1).
    Essencial para a criação de Chaves Primárias (PK) e Estrangeiras (FK) no banco de dados.
    """
    if not url_string: 
        return None
    
    # Extrai o número entre as últimas barras da URL
    return int(url_string.split('/')[-2])

### 3. Extração

Nesta etapa, os dados são consumidos diretamente da API e mantidos no formato próximo ao original (JSON → DataFrame), com mínima transformação.

In [ ]:
# Dicionário para centralizar todos os dados extraídos da API
data = {}

# Iterar sobre todos os endpoints da lista
for item in endpoint:
    data[item] = fetch_all(url, item)

# Armazenar todos os registros em suas devidas variáveis
films_json = data["films"]
people_json = data["people"]
planets_json = data["planets"]
starships_json = data["starships"]

### 4. Transformação e limpeza dos dados 
Nesta etapa, os dados brutos são estruturados em DataFrames e preparados para o modelo relacional. O foco é garantir a qualidade e a tipagem correta para o consumo analítico.

Normalização: Criação da tabela de ligação film_characters para gerenciar o relacionamento entre filmes e personagens.

Limpeza dos dados: Tratamento de campos numéricos que contêm strings como unknown, n/a, ou caracteres especiais (ex: vírgulas e unidades de medida), garantindo que métricas como length sejam puramente numéricas.

Tipagem: Conversão de colunas críticas para int64, otimizando a performance de cálculos e agregados no SQL.

Seleção de Atributos: Filtragem apenas das colunas relevantes para a área de negócio, reduzindo a carga e o uso de memória.


In [ ]:
# 1 - Dataframe (Filmes)
films_df = pd.DataFrame(films_json)[
    ["url", "title", "episode_id", "director", "producer"]
].copy()

# Extração dos IDs dos filmes
films_df['film_id'] = films_df['url'].apply(extract_id)

# Selecionar somente as colunas necessárias
films_df = films_df[["film_id", "title", "episode_id", "director", "producer"]]

# 2 - Dataframe (Personagens que participaram dos filmes)
rows = []
for f in films_json:
    # Extrai o ID do filme uma única vez por iteração do loop
    film_id = extract_id(f["url"]) 
    
    for person_url in f.get("characters", []):
        # Extrai o ID do personagem para cada item da lista
        rows.append({
            "film_id": film_id, 
            "person_id": extract_id(person_url)
        })

film_characters_df = pd.DataFrame(rows)

# 3 - Dataframe (Personagens)
people_df = pd.DataFrame(people_json)[
    ["url", "name", "height", "mass", "gender", "birth_year", "homeworld"]
].copy()

# Tratamento para extrair somente o id do URL
people_df['person_id'] = people_df['url'].apply(extract_id)

# Seleciona somente as colunas necessárias
people_df = people_df[["person_id", "name", "height", "mass", "gender", "birth_year", "homeworld"]]

# 4 - Dataframe (Planetas)
planets_df = pd.DataFrame(planets_json)[
    ["url", "name", "climate", "population", "terrain"]
].copy()

# Tratamento para extrair somente o id do URL
planets_df['planet_id'] = planets_df['url'].apply(extract_id)

# Seleciona somente as colunas necessárias
planets_df = planets_df[["planet_id", "name", "climate", "population", "terrain"]]

# 5 - Dataframe (Naves Espaciais)
starships_df = pd.DataFrame(starships_json)[
    ["url", "name", "model", "starship_class", "max_atmosphering_speed", "length"]
].copy()

# Tratamento para extrair somente o id do URL
starships_df['starship_id'] = starships_df['url'].apply(extract_id)

# Seleciona somente as colunas necessárias
starships_df = starships_df[["starship_id", "name", "model", "starship_class", "max_atmosphering_speed", "length"]]

# Filtra os valores conhecidos da API que não são numéricos
starships_df = starships_df[
    ~starships_df['max_atmosphering_speed'].isin(['n/a', 'unknown'])
]

# Remove letras e caracteres especiais (ex: "1000km" -> "1000")
starships_df['max_atmosphering_speed'] = (
    starships_df['max_atmosphering_speed']
    .str.replace(r'[^0-9]', '', regex=True)
)

# Remove vírgulas (comum em números grandes na SWAPI como '1,600')
starships_df['length'] = starships_df['length'].str.replace(',', '', regex=False)


# Remove o que sobrou de vazio antes de converter para inteiro
starships_df = starships_df[starships_df['max_atmosphering_speed'] != '']

# Conversão final
starships_df['max_atmosphering_speed'] = starships_df['max_atmosphering_speed'].astype('int64')
starships_df['length'] = starships_df['length'].astype('int64')